
# Scripted Trade with Heston

Demonstrate using the Heston model 
- calibration to DAX, STOXX50 and S&P500 options
- pricing: Vanilla Equity Option, Barrier Option, Rainbow Option

Prerequisites:
- Python 3
- pandas, matplotlib
- ORE Python (first 2026 release)

### Run ORE

In [ ]:
from ORE import *
from cantaro import *
import sys, time, math
sys.path.append('..')
import utilities


params = Parameters()
params.fromFile("Input/ore_heston.xml")
#params.fromFile("Input/ore_black.xml")

ore = OREApp(params)
ore.run()

utilities.checkErrorsAndRunTime(ore)

### NPV report

In [ ]:
report = ore.getReport("npv")
utilities.format_report(report)

### Calibration reports

In [ ]:
import pandas as pd

df = pd.read_csv('Output/assetmodel_calibration.csv')
print(df)

df = pd.read_csv('Output/assetmodel_calibration_detail.csv')
print(df)

### Filter one index and expiry

In [ ]:
trade = "RainbowOption"
index = "EQ-RIC:.STOXX50E"
#index = "EQ-RIC:.GDAXI"
#index = "EQ-RIC:.SPX"
expiry = "3M"
df = df[df["#TradeID"] == trade]
df = df[df["Index"] == index]
df = df[df["Expiry"] == expiry]
print(df)

### Plot smile section

In [ ]:
x = df["Moneyness"]
y1 = df["MarketVol"]
y2 = df["ModelVol"]

import matplotlib.pyplot as plt

plt.plot(x, y1, marker="o", linewidth=2, label='Market Vol')
plt.plot(x, y2, marker="3", linewidth=2, label='Model Vol')

plt.xlabel("Moneyness") 
plt.ylabel("Implied Vol") 
plt.title(index + ' Expiry 3M')
plt.legend()

plt.show()

### Get some additional results

In [ ]:
res = pd.read_csv('Output/additional_results.csv')
pd.set_option('display.min_rows', 50)
pd.set_option('display.max_rows', 50)

# filter the trade
res = res[res["#TradeId"] == trade]
print(res[res["ResultId"].str.contains("Heston.Index")], "\n")
print(res[res["ResultId"].str.contains("Heston.Forward")], "\n")


### Create QuantLib Heston Process

In [ ]:
spot = 4337.500000
forward = 4371.906927 
time = 0.25
v0 = 0.024300
kappa = 13.977853
theta = 0.032730
sigma = 1.912942 
rho = -0.638495

zeroYield = 0.0
dividendYield = math.log(forward/spot)/time + zeroYield
drift = dividendYield - zeroYield

asofDate = Date(5, June, 2023)
Settings.instance().evaluationdate = asofDate
dayCount = ActualActual(ActualActual.ISDA)
calendar = TARGET()
spotHandle = QuoteHandle(SimpleQuote(spot))
yieldTS = YieldTermStructureHandle(FlatForward(0, calendar, zeroYield, dayCount))
dividendTS = YieldTermStructureHandle(FlatForward(0, calendar, dividendYield, dayCount))
discretization = HestonProcess.QuadraticExponential

print("feller:  ", 2*theta*kappa/sigma**2)
print("spot:    ", spot)
print("forward: ", spotHandle.value() * yieldTS.discount(time) / dividendTS.discount(time), forward)
print("ln(spot):", np.log(spot))

process =  HestonProcess(yieldTS, dividendTS, spotHandle, v0, kappa, theta, sigma, rho, discretization)

# what's wrong with QuantLib's Heston pdf?
# use cantaro86 for now

### Get path data 

In [ ]:
import numpy as np
from math import sqrt, exp, log

date = "2023-09-05"

res = pd.read_csv('Output/assetmodel_paths.csv')
res = res[res["#TradeId"] == trade]
res = res[res["Index"] == index]
res = res[res["Date"] == date]
# extract the sample values
data = res["Value"].to_numpy(dtype="float32")

# convert to log ( f(t) / splot )
forwardPrice = 4371.906927 
data = np.log(data/spot)

#print(res)
print ("xdata:", data, "\n")

print("length:", len(data))
print("mean:  ", np.mean(data))
print("stdev: ", math.sqrt(np.var(data)))
      

In [ ]:
x = np.linspace(-0.5, 0.3, 200)

plt.hist(data, density=True, bins=100, label="simulation")
plt.plot(x, [Heston_pdf(xx, t=time, v0=v0, mu=drift, theta=theta, sigma=sigma, kappa=kappa, rho=rho) for xx in x], linewidth=3, label="semi-analytic")
plt.xlabel("log(S/S0)") 
plt.ylabel("Density") 
plt.title("")
plt.xlim(-0.5, 0.3)
plt.legend(loc="upper left")
plt.show()